[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/Quantlets/Ch_13/TSA_ch13_explosive_roots/TSA_ch13_explosive_roots.ipynb)

# TSA_ch13_explosive_roots

Right-tailed ADF, SADF (Phillips, Wu and Yu 2011), GSADF and BSADF date-stamping (Phillips, Shi and Yu 2015) on weekly log prices of the Nasdaq 100 (1990-2004), Bitcoin (2014-2023), BET (2000-2012) and the Shanghai Composite (2010-2018), with Monte Carlo critical values.

*Time Series Analysis (TSA), Chapter 13: Speculative bubbles: LPPL models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/TSA/Chapter_13'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# The arch package (Phillips-Perron test of the LPPL residuals) is not preinstalled in Google Colab: pip install arch
import importlib.util, subprocess, sys
if importlib.util.find_spec('arch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import tsa_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the TSA repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
from scipy.signal import lombscargle

SEED = 2026

EPISODES = {'sp500': {'label': 'S&P 500, 2000', 'symbol': 'GSPC.INDX', 'crypto': False, 'low': ('1998-08-01', '1998-11-30'), 'peak': ('1999-06-01', '2000-12-31')}, 'ndx': {'label': 'Nasdaq 100, 2000', 'symbol': 'NDX.INDX', 'crypto': False, 'low': ('1998-08-01', '1998-11-30'), 'peak': ('1999-06-01', '2000-12-31')}, 'bet': {'label': 'BET, 2007', 'symbol': 'BET', 'crypto': False, 'low': ('2006-05-01', '2006-08-31'), 'peak': ('2007-01-01', '2007-12-31')}, 'ssec': {'label': 'Shanghai Composite, 2015', 'symbol': 'SSEC.INDX', 'crypto': False, 'low': ('2014-03-01', '2014-07-31'), 'peak': ('2015-01-01', '2015-12-31')}, 'btc17': {'label': 'Bitcoin, 2017', 'symbol': 'BTC-USD.CC', 'crypto': True, 'low': ('2016-12-01', '2017-03-31'), 'peak': ('2017-06-01', '2018-03-31')}, 'btc21': {'label': 'Bitcoin, 2021', 'symbol': 'BTC-USD.CC', 'crypto': True, 'low': ('2020-08-01', '2020-10-31'), 'peak': ('2021-01-01', '2021-06-30')}}

COLORS = {'sp500': st.COL['sp500'], 'ndx': st.Teal, 'bet': st.Orange, 'ssec': st.Purple, 'btc17': st.COL['btc'], 'btc21': st.COL['btc']}

FIT_LAG_DAYS = 30

PSY_SAMPLES = {'ndx': ('NDX.INDX', False, '1990-01-01', '2004-12-31'), 'btc': ('BTC-USD.CC', True, '2014-09-17', '2023-12-31'), 'bet': ('BET', False, '2000-01-01', '2012-12-31'), 'ssec': ('SSEC.INDX', False, '2010-01-01', '2018-12-31')}

PSY_LABEL = {'ndx': 'Nasdaq 100', 'btc': 'Bitcoin', 'bet': 'BET', 'ssec': 'Shanghai Composite'}

PSY_REPS = 1000

LPPLS_SEARCH = {'m': (0.0, 1.0), 'w': (1.0, 50.0), 'tc_frac': (0.0, 0.3333333333333333), 'damping_min': 1.0}

LPPLS_FILTER = {'m': (0.01, 0.99), 'w': (2.0, 25.0), 'tc_frac': (0.0, 0.2), 'osc_min': 2.5, 'rel_err_max': 0.15, 'lomb_alpha': 0.1, 'ar1_alpha': 0.1}

WINDOWS = list(range(750, 45, -25))

CI_STEP = 5

CRASH, HORIZON = 0.2, 182

CI_LEVEL = 0.2

EVAL_SAMPLES = {'sp500': ('GSPC.INDX', False, '1993-01-01'), 'btc': ('BTC-USD.CC', True, '2014-01-01')}

PARAM_KEYS = ('B', 'm', 'w', 'tc', 'osc', 'damping')

HERE = "."

def prices(symbol, crypto=False):
    """Daily close of a symbol of data/market, with the course conventions (weekdays only and no holiday-filled
    closes, except for crypto assets)."""
    s = read_market(symbol)['close']
    s = pd.to_numeric(s, errors='coerce').dropna()
    s = s[s > 0]
    if not crypto:
        s = s[s.index.dayofweek < 5]
        s = s[s.diff() != 0]
    return s.rename(symbol)


def weekly(s):
    """Weekly log price: last close of each week (Friday)."""
    return np.log(s.resample('W-FRI').last().dropna())


def yrs(idx):
    """Dates as decimal years (the time unit of the LPPL fits)."""
    idx = pd.DatetimeIndex(idx)
    return np.asarray(idx.year + (idx.dayofyear - 1) / 365.25, float)


def todate(x):
    """Decimal year -> date."""
    y = int(np.floor(x))
    return pd.Timestamp(y, 1, 1) + pd.Timedelta(days=float((x - y) * 365.25))


def d2s(d):
    return pd.Timestamp(d).strftime('%Y-%m-%d')


def episode(key):
    """Series, low before the run-up, peak and ex-ante end date of one episode."""
    e = EPISODES[key]
    s = prices(e['symbol'], e['crypto'])
    low = s.loc[e['low'][0]:e['low'][1]].idxmin()
    peak = s.loc[e['peak'][0]:e['peak'][1]].idxmax()
    t2 = s.loc[:peak - pd.Timedelta(days=FIT_LAG_DAYS)].index[-1]
    return s, low, peak, t2


def drawdown(p):
    """Fall from the previous peak: p_t / max_{s<=t} p_s - 1."""
    return p / p.cummax() - 1


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def _cums(y):
    """Cumulative sums for the regression Delta y_t = a + delta y_{t-1} (rows t = 1..n); y may be (n+1,) or (R, n+1)."""
    y = np.atleast_2d(np.asarray(y, float))
    x, z = y[:, :-1], np.diff(y, axis=1)
    pad = lambda a: np.concatenate([np.zeros((a.shape[0], 1)), np.cumsum(a, axis=1)], axis=1)   # noqa: E731
    return pad(x), pad(x * x), pad(z), pad(z * z), pad(x * z)


def _adf_end(C, e, s):
    """t-statistic of delta for the windows [s, e] (s a vector of start rows), for every replication (rows of C)."""
    Sx, Sxx, Sz, Szz, Sxz = (c[:, e + 1][:, None] - c[:, s] for c in C)
    n = (e + 1 - s).astype(float)
    den = n * Sxx - Sx ** 2
    b = (n * Sxz - Sx * Sz) / den
    a = (Sz - b * Sx) / n
    ssr = np.maximum(Szz - a * Sz - b * Sxz, 1e-300)
    return b / np.sqrt(ssr / (n - 2) * n / den)


def adf_window(y):
    """Right-tailed ADF on one window: delta, its standard error and the t-statistic (OLS written out)."""
    y = np.asarray(y, float)
    x, z = y[:-1], np.diff(y)
    X = np.column_stack([np.ones_like(x), x])
    beta, *_ = np.linalg.lstsq(X, z, rcond=None)
    e = z - X @ beta
    s2 = e @ e / (len(z) - 2)
    se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
    return dict(a=float(beta[0]), delta=float(beta[1]), se=float(se), t=float(beta[1] / se), n=len(z))


def min_window(T, r0=None):
    """Smallest window of PSY (2015): r0 = 0.01 + 1.8 / sqrt(T)."""
    r0 = 0.01 + 1.8 / np.sqrt(T) if r0 is None else r0
    return int(np.floor(r0 * T)), r0


def psy(y, r0=None):
    """ADF (whole sample), SADF (sup of forward-expanding ADF), GSADF (sup over start and end points) and the BSADF
    sequence (sup over start points, for each end point) of the log price y."""
    y = np.asarray(y, float)
    n = len(y) - 1
    w0, r0 = min_window(n, r0)
    C = _cums(y)
    bsadf = np.full(n, np.nan)
    fwd = np.full(n, np.nan)
    for e in range(w0 - 1, n):
        stat = _adf_end(C, e, np.arange(0, e - w0 + 2))[0]
        bsadf[e] = stat.max()
        fwd[e] = stat[0]
    return dict(adf=float(fwd[-1]), sadf=float(np.nanmax(fwd)), gsadf=float(np.nanmax(bsadf)), bsadf=bsadf, fwd=fwd,
                w0=w0, r0=r0, n=n)


def psy_cv(T, w0, R=PSY_REPS, seed=SEED, chunk=100):
    """Monte Carlo critical values under a random walk with a weak drift, y_t = T^(-1) + y_{t-1} + e_t (PSY 2015):
    quantiles of ADF, SADF and GSADF, and the 95% quantile of BSADF for each end point."""
    rng = np.random.default_rng(seed)
    Y = np.concatenate([np.zeros((R, 1)), np.cumsum(1.0 / T + rng.standard_normal((R, T)), axis=1)], axis=1)
    bs = np.full((R, T), np.nan)
    fw = np.full((R, T), np.nan)
    for i in range(0, R, chunk):
        C = _cums(Y[i:i + chunk])
        for e in range(w0 - 1, T):
            stat = _adf_end(C, e, np.arange(0, e - w0 + 2))
            bs[i:i + chunk, e] = stat.max(axis=1)
            fw[i:i + chunk, e] = stat[:, 0]
    q = lambda a, p: float(np.quantile(a, p))   # noqa: E731
    return dict(adf={p: q(fw[:, -1], p / 100) for p in (90, 95, 99)},
                sadf={p: q(np.nanmax(fw, axis=1), p / 100) for p in (90, 95, 99)},
                gsadf={p: q(np.nanmax(bs, axis=1), p / 100) for p in (90, 95, 99)},
                bsadf95=np.nanquantile(bs, 0.95, axis=0))


def episodes_above(stat, cv, index, min_len):
    """Date-stamping: periods in which BSADF exceeds its critical value for at least min_len observations."""
    above = np.asarray(stat > cv)
    above[np.isnan(stat) | np.isnan(cv)] = False
    out, i, n = [], 0, len(above)
    while i < n:
        if above[i]:
            j = i
            while j + 1 < n and above[j + 1]:
                j += 1
            if j - i + 1 >= min_len:
                out.append((d2s(index[i]), d2s(index[j]), int(j - i + 1)))
            i = j + 1
        else:
            i += 1
    return out


def run_psy(key):
    """PSY on the weekly log price of one sample: statistics, critical values and dated episodes."""
    sym, crypto, a, b = PSY_SAMPLES[key]
    y = weekly(prices(sym, crypto)).loc[a:b]
    o = psy(y.values)
    cv = psy_cv(o['n'], o['w0'])
    min_len = int(round(np.log(o['n'])))
    ep = episodes_above(o['bsadf'], cv['bsadf95'], y.index[1:], min_len)
    return y, o, cv, ep, min_len


def lppl_design(t, tc, m, w):
    """Columns 1, f, f cos(w ln(tc - t)), f sin(w ln(tc - t)) of the linear part."""
    dt = np.maximum(tc - np.asarray(t, float), 1e-9)
    f = dt ** m
    lg = np.log(dt)
    return np.column_stack([np.ones_like(dt), f, f * np.cos(w * lg), f * np.sin(w * lg)])


def lppl_linear(t, y, tc, m, w):
    """Step 1 of Filimonov-Sornette: for given (tc, m, omega), A, B, C1, C2 by OLS and the sum of squared residuals."""
    X = lppl_design(t, tc, m, w)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    return beta, float(res @ res)


def _damping(m, w, beta):
    C = np.hypot(beta[..., 2], beta[..., 3])
    return m * np.abs(beta[..., 1]) / (w * np.maximum(C, 1e-300))


def _grid(t, y, tcs, ms, ws):
    """SSR of the linear step on a grid of (tc, m, omega), vectorised."""
    TC, M, W = (a.ravel() for a in np.meshgrid(tcs, ms, ws, indexing='ij'))
    dt = np.maximum(TC[:, None] - t[None, :], 1e-9)
    f = dt ** M[:, None]
    lg = np.log(dt)
    X = np.stack([np.ones_like(f), f, f * np.cos(W[:, None] * lg), f * np.sin(W[:, None] * lg)], axis=2)
    XtX = np.einsum('gni,gnj->gij', X, X) + 1e-10 * np.eye(4)
    Xty = np.einsum('gni,n->gi', X, y)
    beta = np.linalg.solve(XtX, Xty[..., None])[..., 0]
    ssr = y @ y - np.einsum('gi,gi->g', beta, Xty)
    return TC, M, W, beta, ssr


def lppl_fit(t, y, search=LPPLS_SEARCH, grid=(8, 8, 16)):
    """Step 2: minimum SSR over the search space (tc in [t2, t2 + (t2 - t1)/3], m in [0, 1], omega in [1, 50],
    damping >= 1): a grid search, then Nelder-Mead from the best grid point. t in years, y = ln p."""
    t = np.asarray(t, float)
    y = np.asarray(y, float)
    t1, t2 = t[0], t[-1]
    D = t2 - t1
    lo = np.array([t2 + search['tc_frac'][0] * D + 1e-6, search['m'][0] + 1e-3, search['w'][0]])
    hi = np.array([t2 + search['tc_frac'][1] * D, search['m'][1] - 1e-3, search['w'][1]])
    TC, M, W, beta, ssr = _grid(t, y, np.linspace(lo[0], hi[0], grid[0]), np.linspace(lo[1], hi[1], grid[1]),
                                np.linspace(lo[2], hi[2], grid[2]))
    ok = _damping(M, W, beta) >= search['damping_min']
    if not ok.any():
        return None
    k = int(np.argmin(np.where(ok, ssr, np.inf)))
    x0 = np.array([TC[k], M[k], W[k]])

    def obj(p):
        q = np.clip(p, lo, hi)
        b, sr = lppl_linear(t, y, *q)
        return sr if _damping(q[1], q[2], b) >= search['damping_min'] else sr + 1e6
    r = optimize.minimize(obj, x0, method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-10, maxiter=400))
    if r.fun < 1e6:
        x0 = np.clip(r.x, lo, hi)
    tc, m, w = x0
    (A, B, C1, C2), s = lppl_linear(t, y, tc, m, w)
    C = np.hypot(C1, C2)
    yhat = lppl_design(t, tc, m, w) @ np.array([A, B, C1, C2])
    return dict(tc=float(tc), m=float(m), w=float(w), A=float(A), B=float(B), C1=float(C1), C2=float(C2), C=float(C),
                ssr=float(s), rmse=float(np.sqrt(s / len(t))), damping=float(m * abs(B) / (w * C)) if C > 0 else np.inf,
                osc=float(w / np.pi * np.log((tc - t1) / (tc - t2))),
                rel_err=float(np.max(np.abs(np.exp(yhat) - np.exp(y)) / np.exp(y))), t1=float(t1), t2=float(t2),
                n=len(t), _t=t, _y=y, _yhat=yhat)


def lomb_pvalue(fit, wmin=2.0, wmax=25.0, nw=200):
    """Lomb test: the detrended residual (tc - t)^(-m) (ln p - A - B (tc - t)^m) against ln(tc - t); probability that
    the highest peak of the normalised periodogram arises by chance."""
    from scipy.signal import lombscargle
    t, y = fit['_t'], fit['_y']
    dt = np.maximum(fit['tc'] - t, 1e-9)
    r = dt ** (-fit['m']) * (y - fit['A'] - fit['B'] * dt ** fit['m'])
    x = np.log(dt)
    r = r - r.mean()
    ws = np.linspace(wmin, wmax, nw)
    p = lombscargle(x, r, ws) / r.var()
    return float(1 - (1 - np.exp(-p.max())) ** min(nw, len(r)))


def ar1_pass(fit, alpha):
    """The residual ln p_hat - ln p is stationary (AR(1)): the Dickey-Fuller and Phillips-Perron tests both reject a
    unit root at level alpha (Chapter 3)."""
    from statsmodels.tsa.stattools import adfuller
    from arch.unitroot import PhillipsPerron
    e = fit['_yhat'] - fit['_y']
    return bool(adfuller(e, maxlag=0, autolag=None, regression='c')[1] < alpha and PhillipsPerron(e, trend='c').pvalue < alpha)


def lppl_conditions(fit, flt=LPPLS_FILTER, search=LPPLS_SEARCH):
    """Filter conditions of Shu and Zhu (2020), eq. (12), for a positive bubble (B < 0). The Lomb test runs only when
    the parameter conditions and the relative error pass, the unit-root test of the residual only when Lomb passes."""
    if fit is None:
        return None
    D = fit['t2'] - fit['t1']
    c = dict(B=fit['B'] < 0, m=flt['m'][0] <= fit['m'] <= flt['m'][1], w=flt['w'][0] <= fit['w'] <= flt['w'][1],
             tc=fit['t2'] + flt['tc_frac'][0] * D <= fit['tc'] <= fit['t2'] + flt['tc_frac'][1] * D,
             osc=fit['osc'] >= flt['osc_min'], damping=fit['damping'] >= search['damping_min'],
             rel_err=fit['rel_err'] <= flt['rel_err_max'])
    if all(c.values()):
        c['lomb'] = lomb_pvalue(fit) <= flt['lomb_alpha']
        c['ar1'] = ar1_pass(fit, flt['ar1_alpha']) if c['lomb'] else False
    else:
        c['lomb'] = c['ar1'] = None
    return c


def qualified(c, which='full'):
    """'full': all the conditions (Shu and Zhu 2020); 'param': only the conditions on the parameters."""
    if c is None:
        return False
    keys = PARAM_KEYS if which == 'param' else tuple(c)
    return all(c[k] is True for k in keys)


def lppl_path(fit, t):
    t = np.asarray(t, float)
    return lppl_design(t, fit['tc'], fit['m'], fit['w']) @ np.array([fit['A'], fit['B'], fit['C1'], fit['C2']])


def fit_window(s, t1, t2):
    """LPPL fit of the log price s between the dates t1 and t2 (inclusive)."""
    x = s.loc[t1:t2]
    return lppl_fit(yrs(x.index), np.log(x.values))


def _ci_point(args):
    """Indicator at one end point: shares of qualified fits over the windows (full filter and parameter filter),
    median critical time of the qualified fits."""
    t, y, i2, windows = args
    full, param, tcs = [], [], []
    for L in windows:
        i1 = i2 - L
        if i1 < 0:
            continue
        f = lppl_fit(t[i1:i2 + 1], y[i1:i2 + 1])
        c = lppl_conditions(f)
        full.append(qualified(c, 'full'))
        param.append(qualified(c, 'param'))
        if qualified(c, 'param'):
            tcs.append(f['tc'])
    return (float(np.mean(full)) if full else np.nan, float(np.mean(param)) if param else np.nan,
            float(np.median(tcs)) if tcs else np.nan)


def confidence_series(s, start, end=None, step=CI_STEP, windows=WINDOWS, procs=None):
    """LPPLS confidence indicator (positive bubbles) for the end points t2 in [start, end], every `step` observations.
    procs=1 runs sequentially (notebooks); otherwise a process pool."""
    t, y = yrs(s.index), np.log(s.values)
    idx = np.arange(s.index.searchsorted(pd.Timestamp(start)), len(s) if end is None else s.index.searchsorted(pd.Timestamp(end), 'right'), step)
    jobs = [(t, y, int(i), windows) for i in idx]
    if procs == 1:
        res = [_ci_point(j) for j in jobs]
    else:
        from multiprocessing import get_context
        with get_context('fork').Pool(procs or max(1, os.cpu_count() - 2)) as pool:
            res = pool.map(_ci_point, jobs, chunksize=4)
    return pd.DataFrame(res, index=s.index[idx], columns=['ci_full', 'ci_param', 'tc_median'])


def cached_ci(name, s, start, end=None, step=CI_STEP, recompute=False):
    """The indicator series of the chapter, cached in Quantlets/Ch_13/ch13_ci_<name>.csv (local copy or GitHub)."""
    path = os.path.join(HERE, f'ch13_ci_{name}.csv')
    if not recompute:
        for p in (path, f'https://raw.githubusercontent.com/danpele/Time-Series-Analysis/main/Quantlets/Ch_13/ch13_ci_{name}.csv'):
            try:
                return pd.read_csv(p, index_col=0, parse_dates=True)
            except Exception:
                pass
    ci = confidence_series(s, start, end, step)
    ci.to_csv(path, float_format='%.4f')
    return ci


def future_fall(s, horizon=HORIZON):
    """For each day t: the largest fall min_{t < u <= t + horizon} p_u / p_t - 1 (NaN when the horizon is not observed)."""
    p = s.values
    d = s.index
    out = np.full(len(s), np.nan)
    j = 0
    for i in range(len(s)):
        lim = d[i] + pd.Timedelta(days=horizon)
        if lim > d[-1]:
            break
        j = max(j, i + 1)
        while j < len(s) and d[j] <= lim:
            j += 1
        out[i] = p[i + 1:j].min() / p[i] - 1 if j > i + 1 else np.nan
    return pd.Series(out, index=d)


def blanchard_watson(T=400, r=0.02, pi=0.98, b0=1.0, sd=0.3, seed=13):
    """Blanchard-Watson bubble: each period it survives with probability pi and then grows by (1 + r)/pi, otherwise it
    bursts (back to noise). E_t[b_{t+1}] = (1 + r) b_t: the expected return equals r."""
    rng = np.random.default_rng(seed)
    b = np.empty(T)
    b[0] = b0
    for t in range(1, T):
        eps = sd * rng.standard_normal()
        b[t] = (1 + r) / pi * b[t - 1] + eps if rng.random() < pi else eps
    return b


def ar1_paths(T=200, rhos=(0.9, 1.0, 1.02), seed=SEED):
    """y_t = rho y_{t-1} + e_t with the same shocks: stationary, unit root and explosive."""
    e = np.random.default_rng(seed).standard_normal(T)
    out = {}
    for rho in rhos:
        y = np.zeros(T)
        for t in range(1, T):
            y[t] = rho * y[t - 1] + e[t]
        out[rho] = y
    return out


def crash_list(s, level=CRASH):
    """Drawdowns of at least 20%: the running peak, the lowest point before the price regains that peak, the size."""
    out = []
    pk_t, pk = s.index[0], s.iloc[0]
    tr_t, tr = pk_t, pk
    for t, p in s.items():
        if p >= pk:
            if tr / pk - 1 <= -level:
                out.append(dict(peak=d2s(pk_t), trough=d2s(tr_t), fall=float(tr / pk - 1)))
            pk_t, pk, tr_t, tr = t, p, t, p
        elif p < tr:
            tr_t, tr = t, p
    if tr / pk - 1 <= -level:
        out.append(dict(peak=d2s(pk_t), trough=d2s(tr_t), fall=float(tr / pk - 1)))
    return out


def hit_rates(d, levels=(0.05, 0.1, 0.2, 0.3, 0.4), col='ci_param'):
    """P(fall >= 20% within 182 days | indicator >= c) against the unconditional frequency."""
    d = d.dropna(subset=['fall'])
    crash = d['fall'] <= -CRASH
    out = dict(base=float(crash.mean()), n=int(len(d)))
    for c in levels:
        a = d[col] >= c
        out[str(c)] = dict(share=float(a.mean()), n=int(a.sum()), hit=float(crash[a].mean()) if a.any() else np.nan)
    return out


def alarm_episodes(d, level=CI_LEVEL, col='ci_param', gap=90):
    """Alarm clusters (days with indicator >= level, separated by fewer than `gap` days) and whether a fall of at
    least 20% follows within 182 days of the first alarm day of each cluster."""
    al = d.index[d[col] >= level]
    out = []
    for t in al:
        if not out or (t - out[-1]['last']).days > gap:
            out.append(dict(first=t, last=t))
        else:
            out[-1]['last'] = t
    for e in out:
        f = d.loc[e['first'], 'fall']
        e['fall'] = None if pd.isna(f) else float(f)
        e['crash'] = None if pd.isna(f) else bool(f <= -CRASH)
    return out


def eval_ci(name):
    """Indicator over the whole sample (S&P 500 since 1993, Bitcoin since 2014) and the fall in the next 182 days."""
    sym, crypto, start = EVAL_SAMPLES[name]
    s = prices(sym, crypto)
    if crypto:
        s = s.loc['2012-01-01':]           # the windows need 750 earlier observations
    ci = cached_ci(f'eval_{name}', s, start, step=7 if crypto else CI_STEP)
    ff = future_fall(s).reindex(ci.index)
    return s, ci.assign(fall=ff)


def episode_ci(key, before=540, after=120):
    s, low, peak, _ = episode(key)
    step = 7 if EPISODES[key]['crypto'] else CI_STEP
    return cached_ci(f'ep_{key}', s, peak - pd.Timedelta(days=before), peak + pd.Timedelta(days=after), step=step), s, peak


def tc_path(key, days_before=240, days_after=30, every=7):
    """Estimated tc as the end of the window t2 moves (start fixed at the low before the run-up)."""
    s, low, peak, _ = episode(key)
    ends = s.loc[peak - pd.Timedelta(days=days_before):peak + pd.Timedelta(days=days_after)].index
    ends = ends[::max(1, every if EPISODES[key]['crypto'] else 5)]
    rows = []
    for t2 in ends:
        f = fit_window(s, low, t2)
        if f is None:
            continue
        rows.append(dict(t2=t2, tc=todate(f['tc']), q=qualified(lppl_conditions(f), 'param')))
    return pd.DataFrame(rows), peak


def windows_at(key):
    """All windows ending at the ex-ante date t2 of one episode: critical time and qualification of each."""
    s, low, peak, t2 = episode(key)
    t, y = yrs(s.index), np.log(s.values)
    i2 = s.index.get_loc(t2)
    rows = []
    for L in WINDOWS:
        f = lppl_fit(t[i2 - L:i2 + 1], y[i2 - L:i2 + 1])
        c = lppl_conditions(f)
        if f is not None:
            rows.append(dict(L=L, tc=todate(f['tc']), q_param=qualified(c, 'param'), q_full=qualified(c, 'full'),
                             **{'c_' + kk: (None if v is None else bool(v)) for kk, v in c.items()}))
    return pd.DataFrame(rows), peak, t2


def worked_examples():
    """Numbers of the worked examples: a rational bubble's growth, explosive AR(1), an LPPL value by hand."""
    A, B, C, m, w, tc, phi = 8.0, -1.2, 0.08, 0.5, 8.0, 2.0, 0.0
    out = {}
    for t in (0.0, 1.0, 1.75, 1.99):
        dt = tc - t
        out[f'{t}'] = dict(dt=dt, fm=dt ** m, lg=float(np.log(dt)), cos=float(np.cos(w * np.log(dt) - phi)),
                           lnp=float(A + B * dt ** m + C * dt ** m * np.cos(w * np.log(dt) - phi)))
    out['p'] = {k: float(np.exp(v['lnp'])) for k, v in out.items()}
    out['bw'] = dict(r=0.02, pi=0.98, g=float(1.02 / 0.98 - 1), dur=float(1 / (1 - 0.98)), half=float(np.log(0.5) / np.log(0.98)))
    return out


def fig_psy_ndx(save_it=True):
    """Nasdaq 100, weekly 1990-2004: log price with the dated episodes, BSADF against its 95% critical values."""
    y, o, cv, ep, min_len = run_psy('ndx')
    idx = y.index[1:]
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw=dict(height_ratios=[1.1, 1]))
    axes[0].plot(y.index, np.exp(y.values), color=st.Teal, label='Nasdaq 100 (weekly close, log scale)')
    axes[0].set_yscale('log')
    for a, b, _ in ep:
        for ax in axes:
            ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=st.IDAred, alpha=0.15, lw=0)
    axes[1].plot(idx, o['bsadf'], color=st.MainBlue, label='BSADF statistic')
    axes[1].plot(idx, cv['bsadf95'], color=st.IDAred, ls='--', label='95% critical value')
    axes[1].set_ylabel('BSADF')
    h, l = axes[0].get_legend_handles_labels()
    h2, l2 = axes[1].get_legend_handles_labels()
    st.fig_legend_bottom(fig, h + h2 + [plt.Rectangle((0, 0), 1, 1, color=st.IDAred, alpha=0.15)],
                         l + l2 + ['dated explosive episode'], ncol=4, y=0.02)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save('tsa_ch13_psy_ndx', save_it)
    # the worked example: at the week of the peak (24 March 2000), the start point that gives the BSADF value
    e = int(y.index[1:].get_indexer([pd.Timestamp('2000-03-24')])[0])
    stat = _adf_end(_cums(y.values), e, np.arange(0, e - o['w0'] + 2))[0]
    s0 = int(np.argmax(stat))
    ex = adf_window(y.values[s0:e + 2])
    ex.update(start=d2s(y.index[s0]), end=d2s(y.index[e + 1]), bsadf=float(stat.max()), cv=float(cv['bsadf95'][e]))
    return dict(T=o['n'], w0=o['w0'], r0=o['r0'], adf=o['adf'], sadf=o['sadf'], gsadf=o['gsadf'],
                cv={k: cv[k] for k in ('adf', 'sadf', 'gsadf')}, episodes=ep, min_len=min_len, example=ex,
                start=d2s(y.index[0]), end=d2s(y.index[-1]))


def fig_psy_panel(save_it=True):
    """Bitcoin, BET and the Shanghai Composite (weekly): log price with dated episodes and BSADF."""
    fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex='col', gridspec_kw=dict(height_ratios=[1.1, 1]))
    out = {}
    for j, (k, c) in enumerate([('btc', st.COL['btc']), ('bet', st.COL['bet']), ('ssec', st.Purple)]):
        y, o, cv, ep, min_len = run_psy(k)
        axes[0, j].plot(y.index, np.exp(y.values), color=c, label=f'{PSY_LABEL[k]} (weekly, log scale)')
        axes[0, j].set_yscale('log')
        axes[0, j].set_title(PSY_LABEL[k])
        axes[1, j].plot(y.index[1:], o['bsadf'], color=st.MainBlue, label='BSADF' if j == 0 else '_')
        axes[1, j].plot(y.index[1:], cv['bsadf95'], color=st.IDAred, ls='--', label='95% critical value' if j == 0 else '_')
        for a, b, _ in ep:
            for ax in axes[:, j]:
                ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=st.IDAred, alpha=0.15, lw=0)
        for ax in axes[:, j]:
            ax.tick_params(axis='x', labelrotation=30)
        out[k] = dict(T=o['n'], gsadf=o['gsadf'], cv95=cv['gsadf'][95], sadf=o['sadf'], sadf95=cv['sadf'][95],
                      episodes=ep, min_len=min_len, start=d2s(y.index[0]), end=d2s(y.index[-1]))
    st.fig_legend_bottom(fig, ncol=6, y=0.02)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save('tsa_ch13_psy_panel', save_it)
    return out

## Run

In [ ]:
print(fig_psy_ndx())
print(fig_psy_panel())

![tsa_ch13_psy_ndx](./tsa_ch13_psy_ndx.png)

Output: `tsa_ch13_psy_ndx.pdf`

![tsa_ch13_psy_panel](./tsa_ch13_psy_panel.png)

Output: `tsa_ch13_psy_panel.pdf`